## Setup: install GeoPandas (once)

This is the same as `%pip install pandas` in Lab 1. It can take a minute or two. If the next cell then says it cannot find geopandas, restart the kernel (the Restart button at the top of the notebook) and run it again. If the install itself fails, show your TA the last red line.

In [ ]:
%pip install geopandas

In [ ]:
import pandas as pd
import geopandas as gpd
import json, os
print("GeoPandas", gpd.__version__, "is ready")

*Importing the actual data*
 to get an idea of how to go about it moving foward 

In [ ]:
gdf = gpd.read_file("../Data/Aire_repartition_populations_caribouForestier.geojson")

Figuring out the size and main content of the file because she huge bro 

In [24]:
print("Total habitat features:", len(gdf))
print(gdf.geom_type.value_counts())
print(gdf["NOM_ANGLA"].dropna().unique()[:20])

Total habitat features: 15
Polygon    15
Name: count, dtype: int64


KeyError: 'NOM_ANGLA'

What organization and cordinate system is used in the data 

In [ ]:
print("CRS:", gdf.crs)
print("Columns:", gdf.columns.tolist())

### Recipe 3. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [ ]:
gdf = gpd.read_file("../Data/Aire_repartition_populations_caribouForestier.geojson")      # or .gpkg, or .geojson
print(gdf.crs, len(gdf), sorted(gdf.geom_type.unique()))
print(gdf.columns.tolist())

gdf= gdf.rename(
    columns={"LEMV": "status", "NOM_POP": "Local_Pop_Name", "NOM_FRANCA": "name_fr", "NOM_ANGLA": "name_en", "NOM_SCIEN": "name_sci", "Rang_S": "Level_Of_Stability", "Tempo": "Years_of_Study", "Sup_KM2": "area_km2", "Sup_HA": "area_ha", "DATE_MAJ": "date_updated"})

gdf = gdf[["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "Years_of_Study", "area_km2", "area_ha", "date_updated", "geometry"]]# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)

gdf["geometry"] = gdf.geometry.simplify(5)
gdf = gdf.to_crs("EPSG:4326")
gdf.to_file("../Data/Caribou_Simplified.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)


EPSG:32198 15 ['Polygon']
['EO_ID', 'NOM_POP', 'NOM_FRANCA', 'AUTRES_NM', 'NOM_ANGLA', 'NOM_SCIEN', 'EOTYPE', 'Rang_S', 'LEMV', 'METHODE', 'Tempo', 'Sup_KM2', 'Sup_HA', 'DATE_MAJ', 'geometry']


In [ ]:
gdf.shape

### The check cell you will reuse for your own data

Change only the four settings at the top when you use it on your project. It reads the file back the way the browser will.

In [10]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Simplified.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.15, -58.94, 55.11  # a box around your area, in degrees
EXPECTED = 15          # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "area_km2", "area_ha"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


Importing my second base data 

In [11]:
gdf_gaspesie = gpd.read_file("../Data/Aire_repartition_caribouMontagnard_Gaspesie.geojson")

Getting the basic infomation on the data

In [ ]:
print("Total habitat features:", len(gdf_gaspesie))
print(gdf_gaspesie.geom_type.value_counts())
print(gdf_gaspesie["NOM_ANGLA"].dropna().unique()[:20])
print("CRS:", gdf_gaspesie.crs)
print("Columns:", gdf_gaspesie.columns.tolist())

### Recipe 3 with data 2. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [ ]:
gdf_gaspesie = gpd.read_file("../Data/Aire_repartition_caribouMontagnard_Gaspesie.geojson")      # or .gpkg, or .geojson
print(gdf_gaspesie.crs, len(gdf_gaspesie), sorted(gdf_gaspesie.geom_type.unique()))
print(gdf_gaspesie.columns.tolist())

gdf_gaspesie = gdf_gaspesie.rename(
    columns={"LEMV": "status", "NOM_POP": "Local_Pop_Name", "NOM_FRANCA": "name_fr", "NOM_ANGLA": "name_en", "NOM_SCIEN": "name_sci", "Rang_S": "Level_Of_Stability", "Tempo": "Years_of_Study", "SUP_KM2": "area_km2", "SUP_HA": "area_ha", "DATE_MAJ": "date_updated"})

gdf_gaspesie = gdf_gaspesie[["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "Years_of_Study", "area_km2", "area_ha", "date_updated", "geometry"]]# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)

gdf_gaspesie["geometry"] = gdf_gaspesie.geometry.simplify(5)
gdf_gaspesie = gdf_gaspesie.to_crs("EPSG:4326")
gdf_gaspesie.to_file("../Data/Caribou_Gaspesie_Simplified.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)


In [ ]:
gdf_gaspesie.shape

### The check cell you will reuse for your own data

Change only the four settings at the top when you use it on your project. It reads the file back the way the browser will.

In [ ]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Gaspesie_Simplified.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.15, -58.94, 55.11  # a box around your area, in degrees
EXPECTED = 1          # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Local_Pop_Name", "name_fr", "name_en", "name_sci", "Level_Of_Stability", "status", "area_km2", "area_ha",]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Polygon']
All checks passed


Working again but with the other data 
The disturbances dataset is available by species only this means I have 6 diffent folders with the data so for this particular data set I will be going through them to seee if they allign and if i can merge them unto one layer.


In [13]:

from pathlib import Path
gdb_paths = sorted(Path("../Data").glob("*.gdb"))
reports = []
for gdb_path in gdb_paths:
        layers = gpd.list_layers(str(gdb_path))
        for layer_name in layers["name"]:
              layer_data = gpd.read_file(
                str(gdb_path),
                layer=layer_name
            )
              reports.append({
    "file": gdb_path.name,
    "layer": layer_name,
    "features": len(layer_data),
    "crs": str(layer_data.crs),
    "geometry_types": ", ".join(sorted(layer_data.geom_type.dropna().unique())),
    "columns": ", ".join(layer_data.columns)
})
comparison = pd.DataFrame(reports)
display(comparison)
        

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


,file,layer,features,crs,geometry_types,columns
0,Charlevoix2020.gdb,Charlevoix_Range_Aire,1,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"HERD, Herd_OS, Shape_Length, Shape_Area, geometry"
1,Charlevoix2020.gdb,Charlevoix2020_Disturb_Perturb_Poly,963,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"Class, Year_added, Comment2020, Shape_Length, ..."
2,Charlevoix2020.gdb,Charlevoix2020_Disturb_Perturb_Line,1077,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiLineString,"Class, Year_added, Shape_Length, geometry"
3,Charlevoix2020.gdb,Charlevoix2020_Anthro_500m,1,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"Shape_Length, Shape_Area, geometry"
4,Charlevoix2020.gdb,Charlevoix2020_Anthro_500m_AndFire_EtFeu,1,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"Shape_Length, Shape_Area, geometry"
5,Manicouagan2020.gdb,Manicouagan_Range_Aire,1,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"HERD, Herd_OS, Shape_Length, Shape_Area, geometry"
6,Manicouagan2020.gdb,Manicouagan2020_Disturb_Perturb_Poly,1124,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"Class, Year_added, Comment2020, Shape_Length, ..."
7,Manicouagan2020.gdb,Manicouagan2020_Disturb_Perturb_Line,1334,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiLineString,"Class, Year_added, Shape_Length, geometry"
8,Manicouagan2020.gdb,Manicouagan2020_Anthro_500m,1,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"Shape_Length, Shape_Area, geometry"
9,Manicouagan2020.gdb,Manicouagan2020_Anthro_500m_AndFire_EtFeu,1,"PROJCS[""Canada_Albers_Equal_Area_Conic"",GEOGCS...",MultiPolygon,"Shape_Length, Shape_Area, geometry"


Creating an identifer for each layer to make sure they dont loose track of the original location and population. As i am going to merge them by layer so I can have them overlay over the map.

Layer: population Range == combined_Range 
Layer : Human disturbances == combined_Hu_Dis
Layer : Linear disturbances (roads, pipelines ect.) == combined_Lin_Dis
Layer: Human disturbances with a 500 m buffer included == combined_Dis_Buff
Layer ; Human disturbances with a 500m buffer plus fires == combined_Dis_Buff_fire 

In [ ]:


combined_Range = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Range_Aire")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Range.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Range, ignore_index=True),
    crs=combined_Range[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [33]:

combined_Hu_Dis = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Disturb_Perturb_Poly")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Hu_Dis.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Hu_Dis, ignore_index=True),
    crs=combined_Hu_Dis[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

Charlevoix2020.gdb ['Charlevoix2020_Disturb_Perturb_Poly']
Manicouagan2020.gdb ['Manicouagan2020_Disturb_Perturb_Poly']
Manouane2020.gdb ['Manouane2020_Disturb_Perturb_Poly']
Pipmuacan2020.gdb ['Pipmuacan2020_Disturb_Perturb_Poly']
Quebec2020.gdb ['Quebec2020_Disturb_Perturb_Poly']
ValdOr2020.gdb ['ValdOr2020_Disturb_Perturb_Poly']
Combined features: 30511
source_dataset
Quebec2020         22783
Pipmuacan2020       2343
Manouane2020        1673
ValdOr2020          1625
Manicouagan2020     1124
Charlevoix2020       963
Name: count, dtype: int64


In [ ]:

combined_Lin_Dis = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Disturb_Perturb_Line")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Lin_Dis.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Lin_Dis, ignore_index=True),
    crs=combined_Lin_Dis[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [ ]:

combined_Dis_Buff = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Anthro_500m")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Dis_Buff.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Dis_Buff, ignore_index=True),
    crs=combined_Dis_Buff[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

In [ ]:

combined_Dis_Buff_fire = []
for gdb_path in gdb_paths:
    layer_info = gpd.list_layers(str(gdb_path))
    matching_layers = [
        name for name in layer_info["name"]
        if name.endswith("_Anthro_500m_AndFire_EtFeu")
    ]
   
    print(gdb_path.name, matching_layers)
    if len(matching_layers) != 1:
        print("Check this file")
        continue

    layer_name = matching_layers[0]
    layer_data = gpd.read_file(
           str(gdb_path),
        layer=layer_name
    )

    layer_data["source_dataset"] = gdb_path.stem
    layer_data["source_layer"] = layer_name
    layer_data["source_feature_id"] = layer_data.index

    combined_Dis_Buff_fire.append(layer_data)

combined = gpd.GeoDataFrame(
    pd.concat(combined_Dis_Buff_fire, ignore_index=True),
    crs=combined_Dis_Buff_fire[0].crs
)

print("Combined features:", len(combined))
print(combined["source_dataset"].value_counts())

Creating the Geojsons for each layer I would like to use in my final project

In [ ]:
combined_range = gpd.GeoDataFrame(
      pd.concat(combined_Range, ignore_index=True),
      crs=combined_Range[0].crs
)
combined_range["geometry"] = combined_range.geometry.simplify(5)
combined_range = combined_range.to_crs("EPSG:4326") 
combined_range.to_file("../Data/Caribou_Range_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

This creates the combined human disturbances data set

In [35]:
combined_hu_dis = gpd.GeoDataFrame(
      pd.concat(combined_Hu_Dis, ignore_index=True),
      crs=combined_Hu_Dis[0].crs
)
combined_hu_dis["geometry"] = combined_hu_dis.geometry.simplify( tolerance=25,
        preserve_topology=True)
combined_hu_dis = combined_hu_dis.to_crs("EPSG:4326") 
combined_hu_dis.to_file("../Data/Caribou_Hu_Dis_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

Simplifying and filtering cause she is a big gall! 

In [57]:
print("Original CRS:", combined_hu_dis.crs),
print("Original features:", len(combined_hu_dis)),

hu_projected = combined_hu_dis.to_crs("EPSG:32198")

columns_to_keep = ["Class", "geometry"]

hu_clean = hu_projected[columns_to_keep].copy()
 
# Use this while the CRS units are metres\n",
hu_clean["geometry"] = hu_clean.geometry.simplify(100, preserve_topology=True) 

hu_clean = hu_clean.to_crs("EPSG:4326")
print(hu_clean.crs) 

hu_clean.to_file("../Data/Cleaned/Caribou_Hu_Dis_Combined_Simplified.geojson",
    driver = "GeoJSON",
    COORDINATE_PRECISION=6,
)
       
print("Cleaned features:", len(hu_clean)),
print("Cleaned columns:", hu_clean.columns.tolist())
hu_clean.geometry.count_coordinates().sum() 

Original CRS: EPSG:4326
Original features: 30511
EPSG:4326
Cleaned features: 30511
Cleaned columns: ['Class', 'geometry']


np.int64(374373)

In [43]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Hu_Dis_Combined_Simplified.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.4  # a box around your area, in degrees
EXPECTED = 30511         # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Class"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['MultiPolygon', 'Polygon']
Not yet:
- The file is 8.5 MB; over about 5 MB the page gets slow. Filter or simplify.


This created the layer for the Linear disturbances. 

In [ ]:
combined_LIN = gpd.GeoDataFrame(
      pd.concat(combined_Lin_Dis, ignore_index=True),
      crs=combined_Lin_Dis[0].crs
)
combined_LIN["geometry"] = combined_LIN.geometry.simplify(5)
combined_LIN = combined_LIN.to_crs("EPSG:4326") 
combined_LIN.to_file("../Data/Caribou_Lin_Dis_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

In [17]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Lin_Dis_Combined.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.7  # a box around your area, in degrees
EXPECTED = 41677         # how many features you expect (rows after your filters)
POPUP_FIELDS = ["Class", "Year_added"]  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['LineString', 'MultiLineString']
Not yet:
- The file is 17.1 MB; over about 5 MB the page gets slow. Filter or simplify.


The combined layer for the disturbances map with 500 m buffer.

In [ ]:
combined_dis_buff = gpd.GeoDataFrame(
      pd.concat(combined_Dis_Buff, ignore_index=True),
      crs=combined_Dis_Buff[0].crs
)
combined_dis_buff["geometry"] = combined_dis_buff.geometry.simplify(5)
combined_dis_buff = combined_dis_buff.to_crs("EPSG:4326") 
combined_dis_buff.to_file("../Data/Caribou_Dis_Buff_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

In [20]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Dis_Buff_Combined.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.7  # a box around your area, in degrees
EXPECTED = 6       # how many features you expect (rows after your filters)
POPUP_FIELDS = []  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['MultiPolygon']
Not yet:
- The file is 11.3 MB; over about 5 MB the page gets slow. Filter or simplify.


The combined layer for anthropogenic disturbances with buffer as well as fire disturbances 

In [ ]:
combined_dis_buff_fire = gpd.GeoDataFrame(
      pd.concat(combined_Dis_Buff_fire, ignore_index=True),
      crs=combined_Dis_Buff_fire[0].crs
)
combined_dis_buff_fire["geometry"] = combined_dis_buff_fire.geometry.simplify(5)
combined_dis_buff_fire = combined_dis_buff_fire.to_crs("EPSG:4326") 
combined_dis_buff_fire.to_file("../Data/Cleaned/Caribou_Dis_Buff_Fire_Combined.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

In [23]:
# ---- settings: change these for your own data ----
FILE = "../Data/Cleaned/Caribou_Dis_Buff_Fire_Combined.geojson"
WEST, EAST, SOUTH, NORTH =  -82.64, 47.329, -58.94, 55.7  # a box around your area, in degrees
EXPECTED = 6       # how many features you expect (rows after your filters)
POPUP_FIELDS = []  # which fields you want to show in the popup
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

DataSourceError: Failed to read GeoJSON data; At line 11, character 9542881: GeoJSON object too complex/large. You may define the OGR_GEOJSON_MAX_OBJ_SIZE configuration option to a value in megabytes to allow for larger features, or 0 to remove any size limit.

In [55]:
Ant = gpd.read_file("../Data/Cleaned/Caribou_Hu_Dis_Combined.geojson")
print("Number of features:", len(Ant))
print("Columns:")
print(Ant.columns.tolist())

print("\nGeometry types:")
print(Ant.geometry.geom_type.value_counts())

print("nFirst few records:")
Ant.head()

Ant["Class"].value_counts()
Ant.geometry.count_coordinates().sum()


Number of features: 30511
Columns:
['Class', 'Year_added', 'Comment2020', 'Shape_Length', 'Shape_Area', 'source_dataset', 'source_layer', 'source_feature_id', 'geometry']

Geometry types:
Polygon         30051
MultiPolygon      460
Name: count, dtype: int64
nFirst few records:


np.int64(1054745)

In [47]:
import geopandas as gpd

gdb_path = "../Data/Charlevoix2020.gdb"

gpd.list_layers(gdb_path)

,name,geometry_type
0,Charlevoix_Range_Aire,MultiPolygon
1,Charlevoix2020_Disturb_Perturb_Poly,MultiPolygon
2,Charlevoix2020_Disturb_Perturb_Line,MultiLineString
3,Charlevoix2020_Anthro_500m,MultiPolygon
4,Charlevoix2020_Anthro_500m_AndFire_EtFeu,MultiPolygon
